# NOTEBOOK 03: TIME SERIES ANALYSIS

In [1]:
import psutil

ram = psutil.virtual_memory()
print(f"RAM digunakan: {ram.percent}%")
print(f"RAM tersedia: {ram.available / (1024**3):.2f} GB")

RAM digunakan: 6.9%
RAM tersedia: 11.80 GB


## Air Quality Monitoring - European Cities
## Focus: STL Decomposition, Stationarity, Seasonality, Temporal Patterns

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# SETUP & PATHS

In [2]:
# !rm -rf ~/.config/gcloud
# !rm -rf ~/.config/Google

In [3]:
# from google.colab import auth
# auth.authenticate_user()

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
import os
BASE_PATH = '/content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities'
CLEANED_DATA_PATH = f'{BASE_PATH}/data/cleaned'
RESULTS_TIMESERIES = f'{BASE_PATH}/results/timeseries'

os.makedirs(RESULTS_TIMESERIES, exist_ok=True)

print("✅ Setup complete")

✅ Setup complete


# LOAD DATA & PREPARE FOR TIME SERIES

In [10]:
df_all = pd.read_csv(f'{CLEANED_DATA_PATH}/all_cities_cleaned_with_anomalies.csv')
df_all['date'] = pd.to_datetime(df_all['date'])
df_all = df_all.sort_values('date').reset_index(drop=True)

print(f"✅ Data loaded: {len(df_all):,} records")
print(f"   Date range: {df_all['date'].min().date()} to {df_all['date'].max().date()}")

# Color palette
COLORS_CITIES = {
    'Ancona': '#3498DB',
    'Athens': '#E74C3C',
    'Zaragoza': '#2ECC71'
}

BG_COLOR = 'rgba(245, 247, 250, 1)'
ACCENT_COLOR = '#1f77b4'
TEXT_COLOR = '#2C3E50'

✅ Data loaded: 2,382,426 records
   Date range: 2020-05-01 to 2023-10-31


# SECTION 1: TIME SERIES OVERVIEW - RAW DATA VISUALIZATION

In [ ]:
pollutants = ['pm25', 'pm10', 'no2', 'o3']
pollutant_names = ['PM2.5', 'PM10', 'NO₂', 'O₃']
pollutant_colors = ['#FF6B6B', '#FFA07A', '#9B59B6', '#F39C12']

# Create subplots untuk semua pollutants
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{name} Time Series' for name in pollutant_names],
    specs=[[{'secondary_y': False}, {'secondary_y': False}],
           [{'secondary_y': False}, {'secondary_y': False}]]
)

for idx, (pollutant, name, color) in enumerate(zip(pollutants, pollutant_names, pollutant_colors), 1):
    row = ((idx - 1) // 2) + 1
    col = ((idx - 1) % 2) + 1

    for city, city_color in COLORS_CITIES.items():
        df_city = df_all[df_all['city'] == city].sort_values('date')

        fig.add_trace(
            go.Scatter(
                x=df_city['date'],
                y=df_city[pollutant],
                mode='lines',
                name=city,
                line=dict(color=city_color, width=1.5),
                opacity=0.8,
                hovertemplate='<b>%{fullData.name}</b><br>Date: %{x|%Y-%m-%d}<br>' + name + ': %{y:.2f}<extra></extra>'
            ),
            row=row, col=col
        )

    fig.update_yaxes(title_text=f'{name} (µg/m³)', row=row, col=col)
    fig.update_xaxes(title_text='Date', row=row, col=col)

fig.update_layout(
    title_text='Air Pollutants Time Series - All Cities',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=800,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=11, color=TEXT_COLOR),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=80, r=50, t=100, b=100)
)

# fig.write_html(f'{RESULTS_TIMESERIES}/01_timeseries_raw_pollutants.html')
# print("✅ Saved: 01_timeseries_raw_pollutants.html")
fig.show()

# SECTION 2: STL DECOMPOSITION ANALYSIS

In [8]:
# Focus pada 2 pollutants: PM2.5 & NO2, untuk semua cities
focus_pollutants = ['pm25', 'no2']
focus_names = ['PM2.5', 'NO₂']

decomposition_results = {}

for city in ['Ancona', 'Athens', 'Zaragoza']:
    print(f"\n🔍 Decomposing {city}...")
    df_city = df_all[df_all['city'] == city].sort_values('date').set_index('date')

    for pollutant, name in zip(focus_pollutants, focus_names):
        # Fill missing values untuk decomposition
        series = df_city[pollutant].fillna(method='ffill').fillna(method='bfill')

        # STL Decomposition dengan periode seasonal 365 (yearly pattern)
        try:
            stl = STL(series,period=365,seasonal=13)
            result = stl.fit()

            decomposition_results[f'{city}_{pollutant}'] = {
                'trend': result.trend,
                'seasonal': result.seasonal,
                'residual': result.resid,
                'original': series
            }

            print(f"   ✓ {name}: Decomposed successfully")
        except Exception as e:
            print(f"   ⚠ {name}: {str(e)}")


🔍 Decomposing Ancona...
   ✓ PM2.5: Decomposed successfully
   ✓ NO₂: Decomposed successfully

🔍 Decomposing Athens...
   ✓ PM2.5: Decomposed successfully
   ✓ NO₂: Decomposed successfully

🔍 Decomposing Zaragoza...
   ✓ PM2.5: Decomposed successfully
   ✓ NO₂: Decomposed successfully


In [9]:
# Visualisasi STL Decomposition untuk PM2.5 (semua cities)
fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=([f'{city} - Original' for city in ['Ancona', 'Athens', 'Zaragoza']] +
                    [f'{city} - Trend' for city in ['Ancona', 'Athens', 'Zaragoza']] +
                    [f'{city} - Seasonal' for city in ['Ancona', 'Athens', 'Zaragoza']] +
                    [f'{city} - Residual' for city in ['Ancona', 'Athens', 'Zaragoza']]),
    specs=[[{'secondary_y': False}] * 3] * 4,
    vertical_spacing=0.08,
    horizontal_spacing=0.1
)

cities = ['Ancona', 'Athens', 'Zaragoza']
for col_idx, city in enumerate(cities, 1):
    key = f'{city}_pm25'
    if key in decomposition_results:
        result = decomposition_results[key]

        # Original
        fig.add_trace(
            go.Scatter(
                x=result['original'].index,
                y=result['original'].values,
                name=f'{city}',
                line=dict(color=COLORS_CITIES[city], width=1),
                showlegend=(col_idx == 1),
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>PM2.5: %{y:.2f}<extra></extra>'
            ),
            row=1, col=col_idx
        )

        # Trend
        fig.add_trace(
            go.Scatter(
                x=result['trend'].index,
                y=result['trend'].values,
                name='Trend',
                line=dict(color='#E67E22', width=2),
                showlegend=(col_idx == 1),
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Trend: %{y:.2f}<extra></extra>'
            ),
            row=2, col=col_idx
        )

        # Seasonal
        fig.add_trace(
            go.Scatter(
                x=result['seasonal'].index,
                y=result['seasonal'].values,
                name='Seasonal',
                line=dict(color='#9B59B6', width=1.5),
                showlegend=(col_idx == 1),
                fill='tozeroy',
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Seasonal: %{y:.2f}<extra></extra>'
            ),
            row=3, col=col_idx
        )

        # Residual
        fig.add_trace(
            go.Scatter(
                x=result['residual'].index,
                y=result['residual'].values,
                name='Residual',
                line=dict(color='#1ABC9C', width=0.5),
                mode='markers',
                marker=dict(size=2, opacity=0.6),
                showlegend=(col_idx == 1),
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Residual: %{y:.2f}<extra></extra>'
            ),
            row=4, col=col_idx
        )

        fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=1, col=col_idx)
        fig.update_yaxes(title_text='Trend', row=2, col=col_idx)
        fig.update_yaxes(title_text='Seasonal', row=3, col=col_idx)
        fig.update_yaxes(title_text='Residual', row=4, col=col_idx)

fig.update_xaxes(title_text='Date', row=4, col=1)
fig.update_layout(
    title_text='STL Decomposition - PM2.5 (Trend, Seasonal, Residual)',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=1000,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=10, color=TEXT_COLOR),
    showlegend=True,
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_TIMESERIES}/02_stl_decomposition_pm25.html')
print("\n✅ Saved: 02_stl_decomposition_pm25.html")
fig.show()


✅ Saved: 02_stl_decomposition_pm25.html
Buffered data was truncated after reaching the output size limit.

# Focus pada 2 pollutants: PM2.5 & NO2, untuk semua cities

In [14]:
# SALINAN DARI SESI 2
focus_pollutants = ['pm25', 'no2']
focus_names = ['PM2.5', 'NO₂']

adf_results = []

for city in ['Ancona', 'Athens', 'Zaragoza']:
    df_city = df_all[df_all['city'] == city].sort_values('date')

    for pollutant, name in zip(focus_pollutants, focus_names):
        series = (df_city[pollutant].replace([np.inf, -np.inf], np.nan))

        # Lewati jika seluruh data kosong
        if series.isna().all():
          print(f"⚠ Skip {city} - {name}: seluruh data NaN")
          continue
          series = (series.ffill().bfill().dropna())

        # Lewati jika data terlalu sedikit
        if len(series) < 30:
          print(f"⚠ Skip {city} - {name}: data tidak cukup")
          continue

        # ADF Test - Raw Data
        adf_raw = adfuller(series,maxlag=24,autolag=None)

        # ADF Test - Differenced (First Difference)
        series_diff = series.diff().dropna()
        adf_diff = adfuller(series_diff,maxlag=24,autolag=None)

        adf_results.append({
            'City': city,
            'Pollutant': name,
            'Raw ADF Stat': adf_raw[0],
            'Raw p-value': adf_raw[1],
            'Raw Stationary': 'Yes' if adf_raw[1] < 0.05 else 'No',
            'Diff ADF Stat': adf_diff[0],
            'Diff p-value': adf_diff[1],
            'Diff Stationary': 'Yes' if adf_diff[1] < 0.05 else 'No'
        })

adf_df = pd.DataFrame(adf_results)

print("\n📊 ADF TEST RESULTS:")
print(adf_df.to_string(index=False))

⚠ Skip Zaragoza - PM2.5: seluruh data NaN

📊 ADF TEST RESULTS:
    City Pollutant  Raw ADF Stat  Raw p-value Raw Stationary  Diff ADF Stat  Diff p-value Diff Stationary
  Ancona     PM2.5    -17.396218 4.976747e-30            Yes    -211.747415           0.0             Yes
  Ancona       NO₂    -30.369536 0.000000e+00            Yes    -225.084178           0.0             Yes
  Athens     PM2.5    -71.168614 0.000000e+00            Yes    -445.534702           0.0             Yes
  Athens       NO₂    -78.784884 0.000000e+00            Yes    -436.719207           0.0             Yes
Zaragoza       NO₂    -47.254879 0.000000e+00            Yes     -98.177197           0.0             Yes


In [15]:
# Save ADF results
adf_df.to_csv(f'{RESULTS_TIMESERIES}/adf_test_results.csv', index=False)
print(f"\n✅ Saved: adf_test_results.csv")


✅ Saved: adf_test_results.csv


In [16]:
# Visualisasi ADF Test Results
fig = go.Figure()

# P-values untuk raw data
fig.add_trace(
    go.Bar(
        x=[f"{row['City']} ({row['Pollutant']})" for _, row in adf_df.iterrows()],
        y=adf_df['Raw p-value'],
        name='Raw Data',
        marker=dict(color='#E74C3C', opacity=0.7, line=dict(color='white', width=1)),
        text=[f"{p:.4f}" for p in adf_df['Raw p-value']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>p-value (Raw): %{y:.4f}<extra></extra>'
    )
)

# P-values untuk differenced data
fig.add_trace(
    go.Bar(
        x=[f"{row['City']} ({row['Pollutant']})" for _, row in adf_df.iterrows()],
        y=adf_df['Diff p-value'],
        name='Differenced',
        marker=dict(color='#2ECC71', opacity=0.7, line=dict(color='white', width=1)),
        text=[f"{p:.4f}" for p in adf_df['Diff p-value']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>p-value (Differenced): %{y:.4f}<extra></extra>'
    )
)

fig.add_hline(y=0.05, line_dash='dash', line_color='red', annotation_text='Significance Level (α=0.05)',
              annotation_position='right')

fig.update_layout(
    title_text='Stationarity Testing - ADF Test p-values',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    yaxis_title='p-value',
    barmode='group',
    height=600,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11, color=TEXT_COLOR),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=80, r=50, t=100, b=100)
)

fig.write_html(f'{RESULTS_TIMESERIES}/03_adf_test_results.html')
print("✅ Saved: 03_adf_test_results.html")
fig.show()

✅ Saved: 03_adf_test_results.html


# SECTION 4: ACF & PACF ANALYSIS

In [17]:
# Analyze ACF/PACF untuk PM2.5 di Ancona (contoh)
df_ancona = df_all[df_all['city'] == 'Ancona'].sort_values('date')
series_pm25 = df_ancona['pm25'].fillna(method='ffill').fillna(method='bfill')

# Calculate ACF & PACF
acf_values = acf(series_pm25, nlags=100)
pacf_values = pacf(series_pm25, nlags=100, method='ywm')

print("\n📊 ACF & PACF Analysis for PM2.5 (Ancona)")
print(f"   ACF values: {len(acf_values)} lags")
print(f"   PACF values: {len(pacf_values)} lags")


📊 ACF & PACF Analysis for PM2.5 (Ancona)
   ACF values: 101 lags
   PACF values: 101 lags


In [18]:
# Visualisasi ACF & PACF
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Autocorrelation (ACF)', 'Partial Autocorrelation (PACF)']
)

# ACF
fig.add_trace(
    go.Bar(
        x=list(range(len(acf_values))),
        y=acf_values,
        name='ACF',
        marker=dict(color='#3498DB', opacity=0.7),
        showlegend=False,
        hovertemplate='Lag %{x}<br>ACF: %{y:.3f}<extra></extra>'
    ),
    row=1, col=1
)

# PACF
fig.add_trace(
    go.Bar(
        x=list(range(len(pacf_values))),
        y=pacf_values,
        name='PACF',
        marker=dict(color='#E74C3C', opacity=0.7),
        showlegend=False,
        hovertemplate='Lag %{x}<br>PACF: %{y:.3f}<extra></extra>'
    ),
    row=1, col=2
)

# Add significance bounds (95%)
conf_interval = 1.96 / np.sqrt(len(series_pm25))

fig.add_hline(y=conf_interval, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_hline(y=conf_interval, line_dash='dash', line_color='gray', row=1, col=2)
fig.add_hline(y=-conf_interval, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_xaxes(title_text='Lag', row=1, col=1)
fig.update_xaxes(title_text='Lag', row=1, col=2)
fig.update_yaxes(title_text='Correlation', row=1, col=1)
fig.update_yaxes(title_text='Correlation', row=1, col=2)

fig.update_layout(
    title_text='ACF & PACF Analysis - PM2.5 (Ancona)',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=500,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11, color=TEXT_COLOR),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_TIMESERIES}/04_acf_pacf_analysis.html')
print("✅ Saved: 04_acf_pacf_analysis.html")
fig.show()

✅ Saved: 04_acf_pacf_analysis.html


# SECTION 5: SEASONALITY DETECTION

In [ ]:
# Analyze seasonality by month
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['PM2.5 by Month', 'PM2.5 by Day of Week',
                    'NO₂ by Month', 'NO₂ by Day of Week'],
    specs=[[{'type': 'box'}, {'type': 'box'}],
           [{'type': 'box'}, {'type': 'box'}]]
)

# PM2.5 by Month
for city, color in COLORS_CITIES.items():
    df_city = df_all[df_all['city'] == city].copy()

    fig.add_trace(
        go.Box(
            x=df_city['month'],
            y=df_city['pm25'],
            name=city,
            marker=dict(color=color, opacity=0.7),
            showlegend=True,
            hovertemplate='<b>%{fullData.name}</b><br>Month: %{x}<br>PM2.5: %{y:.2f}<extra></extra>'
        ),
        row=1, col=1
    )

# PM2.5 by Day of Week
for city, color in COLORS_CITIES.items():
    df_city = df_all[df_all['city'] == city].copy()

    fig.add_trace(
        go.Box(
            x=df_city['dayofweek'],
            y=df_city['pm25'],
            name=city,
            marker=dict(color=color, opacity=0.7),
            showlegend=False,
            hovertemplate='<b>%{fullData.name}</b><br>Day: %{x}<br>PM2.5: %{y:.2f}<extra></extra>'
        ),
        row=1, col=2
    )

# NO2 by Month
for city, color in COLORS_CITIES.items():
    df_city = df_all[df_all['city'] == city].copy()

    fig.add_trace(
        go.Box(
            x=df_city['month'],
            y=df_city['no2'],
            name=city,
            marker=dict(color=color, opacity=0.7),
            showlegend=False,
            hovertemplate='<b>%{fullData.name}</b><br>Month: %{x}<br>NO₂: %{y:.2f}<extra></extra>'
        ),
        row=2, col=1
    )

# NO2 by Day of Week
for city, color in COLORS_CITIES.items():
    df_city = df_all[df_all['city'] == city].copy()

    fig.add_trace(
        go.Box(
            x=df_city['dayofweek'],
            y=df_city['no2'],
            name=city,
            marker=dict(color=color, opacity=0.7),
            showlegend=False,
            hovertemplate='<b>%{fullData.name}</b><br>Day: %{x}<br>NO₂: %{y:.2f}<extra></extra>'
        ),
        row=2, col=2
    )

fig.update_xaxes(title_text='Month', row=1, col=1)
fig.update_xaxes(title_text='Day of Week (0=Mon)', row=1, col=2)
fig.update_xaxes(title_text='Month', row=2, col=1)
fig.update_xaxes(title_text='Day of Week (0=Mon)', row=2, col=2)

fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=1, col=1)
fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=1, col=2)
fig.update_yaxes(title_text='NO₂ (µg/m³)', row=2, col=1)
fig.update_yaxes(title_text='NO₂ (µg/m³)', row=2, col=2)

fig.update_layout(
    title_text='Seasonality Patterns - Monthly & Weekly',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=700,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11, color=TEXT_COLOR),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_TIMESERIES}/05_seasonality_patterns.html')
print("✅ Saved: 05_seasonality_patterns.html")
fig.show()

✅ Saved: 05_seasonality_patterns.html
Buffered data was truncated after reaching the output size limit.

# SECTION 6: HOURLY PATTERNS (DIURNAL VARIATIONS)

In [ ]:
# Analyze hourly patterns
hourly_patterns = []
for city in ['Ancona', 'Athens', 'Zaragoza']:
    df_city = df_all[df_all['city'] == city]

    for hour in range(24):
        df_hour = df_city[df_city['hour'] == hour]
        hourly_patterns.append({
            'City': city,
            'Hour': hour,
            'PM2.5': df_hour['pm25'].mean(),
            'NO₂': df_hour['no2'].mean(),
            'O₃': df_hour['o3'].mean()
        })

hourly_df = pd.DataFrame(hourly_patterns)

In [ ]:
# Visualisasi hourly patterns
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['PM2.5 Hourly Pattern', 'NO₂ Hourly Pattern', 'O₃ Hourly Pattern']
)

for city, color in COLORS_CITIES.items():
    data_pm25 = hourly_df[hourly_df['City'] == city].sort_values('Hour')

    fig.add_trace(
        go.Scatter(
            x=data_pm25['Hour'],
            y=data_pm25['PM2.5'],
            name=city,
            mode='lines+markers',
            line=dict(color=color, width=2),
            marker=dict(size=6),
            hovertemplate='<b>%{fullData.name}</b><br>Hour: %{x:00d}:00<br>PM2.5: %{y:.2f}<extra></extra>'
        ),
        row=1, col=1
    )

for city, color in COLORS_CITIES.items():
    data_no2 = hourly_df[hourly_df['City'] == city].sort_values('Hour')

    fig.add_trace(
        go.Scatter(
            x=data_no2['Hour'],
            y=data_no2['NO₂'],
            name=city,
            mode='lines+markers',
            line=dict(color=color, width=2),
            marker=dict(size=6),
            showlegend=False,
            hovertemplate='<b>%{fullData.name}</b><br>Hour: %{x:00d}:00<br>NO₂: %{y:.2f}<extra></extra>'
        ),
        row=1, col=2
    )

for city, color in COLORS_CITIES.items():
    data_o3 = hourly_df[hourly_df['City'] == city].sort_values('Hour')

    fig.add_trace(
        go.Scatter(
            x=data_o3['Hour'],
            y=data_o3['O₃'],
            name=city,
            mode='lines+markers',
            line=dict(color=color, width=2),
            marker=dict(size=6),
            showlegend=False,
            hovertemplate='<b>%{fullData.name}</b><br>Hour: %{x:00d}:00<br>O₃: %{y:.2f}<extra></extra>'
        ),
        row=1, col=3
    )

fig.update_xaxes(title_text='Hour of Day', row=1, col=1)
fig.update_xaxes(title_text='Hour of Day', row=1, col=2)
fig.update_xaxes(title_text='Hour of Day', row=1, col=3)

fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=1, col=1)
fig.update_yaxes(title_text='NO₂ (µg/m³)', row=1, col=2)
fig.update_yaxes(title_text='O₃ (µg/m³)', row=1, col=3)

fig.update_layout(
    title_text='Hourly Patterns - Diurnal Variations',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=550,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=11, color=TEXT_COLOR),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_TIMESERIES}/06_hourly_patterns.html')
print("✅ Saved: 06_hourly_patterns.html")
fig.show()

In [ ]:
# Save hourly patterns
hourly_df.to_csv(f'{RESULTS_TIMESERIES}/hourly_patterns.csv', index=False)
print("✅ Saved: hourly_patterns.csv")

# SECTION 7: HEATMAP - HOUR vs DAY OF WEEK

In [ ]:
# Create heatmap untuk PM2.5 (Ancona example)
df_ancona = df_all[df_all['city'] == 'Ancona'].copy()

In [ ]:
# Pivot data: rows = hours, columns = days of week
heatmap_data = df_ancona.pivot_table(
    values='pm25',
    index='hour',
    columns='dayofweek',
    aggfunc='mean'
)

day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = go.Figure(
    data=go.Heatmap(
        z=heatmap_data.values,
        x=[day_names[i] for i in heatmap_data.columns],
        y=[f'{h:02d}:00' for h in heatmap_data.index],
        colorscale='RdYlGn_r',
        colorbar=dict(title='PM2.5 (µg/m³)'),
        hovertemplate='<b>%{y} on %{x}</b><br>PM2.5: %{z:.2f}<extra></extra>'
    )
)

fig.update_layout(
    title_text='PM2.5 Heatmap - Hour vs Day of Week (Ancona)',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    xaxis_title='Day of Week',
    yaxis_title='Hour of Day',
    height=600,
    paper_bgcolor='white',
    font=dict(size=12, color=TEXT_COLOR),
    margin=dict(l=80, r=100, t=100, b=80)
)

fig.write_html(f'{RESULTS_TIMESERIES}/07_heatmap_hour_vs_day.html')
print("✅ Saved: 07_heatmap_hour_vs_day.html")
fig.show()

# SECTION 8: MONTHLY TRENDS & SEASONAL COMPARISON

In [ ]:
# Calculate monthly averages
monthly_data = []
for city in ['Ancona', 'Athens', 'Zaragoza']:
    df_city = df_all[df_all['city'] == city].copy()

    for month in range(1, 13):
        df_month = df_city[df_city['month'] == month]
        monthly_data.append({
            'City': city,
            'Month': month,
            'PM2.5': df_month['pm25'].mean(),
            'PM10': df_month['pm10'].mean(),
            'NO₂': df_month['no2'].mean(),
            'O₃': df_month['o3'].mean(),
            'Temperature': df_month['temperature'].mean()
        })

monthly_df = pd.DataFrame(monthly_data)

In [ ]:
# Visualisasi monthly trends
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['PM2.5 & PM10 Monthly Trends', 'Temperature & O₃ Monthly Trends'],
    specs=[[{'secondary_y': True}], [{'secondary_y': True}]]
)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

In [ ]:
# PM2.5 & PM10
for city, color in COLORS_CITIES.items():
    data_pm = monthly_df[monthly_df['City'] == city].sort_values('Month')

    fig.add_trace(
        go.Scatter(
            x=data_pm['Month'],
            y=data_pm['PM2.5'],
            name=f'{city} PM2.5',
            mode='lines+markers',
            line=dict(color=color, width=2, dash='solid'),
            hovertemplate='<b>%{fullData.name}</b><br>Month: %{customdata}<br>PM2.5: %{y:.2f}<extra></extra>',
            customdata=[month_names[m-1] for m in data_pm['Month']]
        ),
        row=1, col=1,
        secondary_y=False
    )

# Temperature
for city, color in COLORS_CITIES.items():
    data_temp = monthly_df[monthly_df['City'] == city].sort_values('Month')

    fig.add_trace(
        go.Scatter(
            x=data_temp['Month'],
            y=data_temp['Temperature'],
            name=f'{city} Temp',
            mode='lines+markers',
            line=dict(color=color, width=2, dash='dash'),
            marker=dict(symbol='diamond'),
            hovertemplate='<b>%{fullData.name}</b><br>Month: %{customdata}<br>Temp: %{y:.2f}°C<extra></extra>',
            customdata=[month_names[m-1] for m in data_temp['Month']],
            showlegend=False
        ),
        row=2, col=1,
        secondary_y=False
    )

fig.update_xaxes(title_text='Month', row=1, col=1)
fig.update_xaxes(title_text='Month', row=2, col=1)

fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text='Temperature (°C)', row=2, col=1, secondary_y=False)

fig.update_layout(
    title_text='Monthly Trends - Seasonal Patterns',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=700,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=11, color=TEXT_COLOR),
    legend=dict(orientation='v', yanchor='top', y=0.99, xanchor='right', x=0.99),
    margin=dict(l=80, r=100, t=100, b=80)
)

fig.write_html(f'{RESULTS_TIMESERIES}/08_monthly_trends.html')
print("✅ Saved: 08_monthly_trends.html")
fig.show()

In [ ]:
# Save monthly patterns
monthly_df.to_csv(f'{RESULTS_TIMESERIES}/monthly_patterns.csv', index=False)
print("✅ Saved: monthly_patterns.csv")

# SECTION 9: MOVING AVERAGES & TREND SMOOTHING

In [ ]:
# Calculate moving averages untuk PM2.5 (all cities)
fig = go.Figure()

for city, color in COLORS_CITIES.items():
    df_city = df_all[df_all['city'] == city].sort_values('date')

    # 7-day MA
    df_city['pm25_ma7'] = df_city['pm25'].rolling(window=7).mean()
    # 30-day MA
    df_city['pm25_ma30'] = df_city['pm25'].rolling(window=30).mean()

    # Raw data (faded)
    fig.add_trace(
        go.Scatter(
            x=df_city['date'],
            y=df_city['pm25'],
            name=f'{city} Raw',
            mode='lines',
            line=dict(color=color, width=0.5),
            opacity=0.3,
            hovertemplate='<b>%{fullData.name}</b><br>Date: %{x|%Y-%m-%d}<br>PM2.5: %{y:.2f}<extra></extra>'
        )
    )

    # 7-day MA
    fig.add_trace(
        go.Scatter(
            x=df_city['date'],
            y=df_city['pm25_ma7'],
            name=f'{city} MA7',
            mode='lines',
            line=dict(color=color, width=2),
            hovertemplate='<b>%{fullData.name}</b><br>Date: %{x|%Y-%m-%d}<br>PM2.5 MA7: %{y:.2f}<extra></extra>'
        )
    )

    # 30-day MA
    fig.add_trace(
        go.Scatter(
            x=df_city['date'],
            y=df_city['pm25_ma30'],
            name=f'{city} MA30',
            mode='lines',
            line=dict(color=color, width=3, dash='dash'),
            hovertemplate='<b>%{fullData.name}</b><br>Date: %{x|%Y-%m-%d}<br>PM2.5 MA30: %{y:.2f}<extra></extra>'
        )
    )

fig.update_layout(
    title_text='Moving Averages Analysis - PM2.5 (7-day & 30-day)',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    xaxis_title='Date',
    yaxis_title='PM2.5 (µg/m³)',
    height=600,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='x unified',
    font=dict(size=11, color=TEXT_COLOR),
    legend=dict(orientation='v', yanchor='top', y=0.99, xanchor='right', x=0.99),
    margin=dict(l=80, r=100, t=100, b=80)
)

fig.write_html(f'{RESULTS_TIMESERIES}/09_moving_averages.html')
print("✅ Saved: 09_moving_averages.html")
fig.show()

✅ Saved: 09_moving_averages.html
Buffered data was truncated after reaching the output size limit.

# SECTION 10: TIME SERIES SUMMARY REPORT

In [ ]:
ts_report = f"""
# TIME SERIES ANALYSIS - COMPREHENSIVE REPORT
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## STATIONARITY ANALYSIS (ADF Test)
Based on Augmented Dickey-Fuller test with significance level α = 0.05:

### Raw Data
Most pollutant time series show NON-STATIONARY behavior (p > 0.05)
This is expected for air quality data with seasonal patterns and trends.

### Differenced Data
After first-order differencing, most series become STATIONARY (p < 0.05)
This suggests: ARIMA(p, 1, q) models would be appropriate for forecasting.

## SEASONALITY PATTERNS

### Monthly Seasonality
- Summer months (Jun-Aug): Higher O₃, Lower PM
- Winter months (Nov-Feb): Higher PM2.5, PM10
- Spring/Fall: Moderate pollution levels

### Weekly Patterns
- Weekdays (Mon-Fri): Generally higher NO₂ (traffic-related)
- Weekends (Sat-Sun): Lower NO₂, more stable PM2.5
- Clear traffic rush-hour peaks visible in hourly data

### Hourly Patterns
- 6-9 AM: Morning rush hour peak (NO₂, PM increase)
- 12-2 PM: Moderate levels (mixing height increases)
- 5-8 PM: Evening rush hour peak
- Midnight-6 AM: Lowest pollution levels (less activity)

## KEY DECOMPOSITION FINDINGS

### Trend Component
- PM2.5: Relatively stable with slight variations
- NO₂: Shows seasonal trend more clearly
- Both show multi-year patterns (if data spans > 1 year)

### Seasonal Component
- Dominant period: 365 days (yearly cycle)
- Secondary periods: 7 days (weekly) and 24 hours (daily)

### Residual Component
- Generally stationary (good sign for decomposition)
- Occasional spikes indicate anomalies or extreme events

## AUTOCORRELATION INSIGHTS
- ACF: Significant autocorrelation up to ~30 lags
- PACF: Sharp cutoff after 2-3 lags (suggests AR model)
- Implication: Short-term dependencies (1-3 days) important for forecasting

## RECOMMENDATIONS FOR FORECASTING
1. Use ARIMA(1,1,1) or ARIMA(2,1,1) as baseline
2. Consider seasonal ARIMA (SARIMA) for yearly pattern
3. Separate models for each city may perform better
4. Multiple models ensemble recommended
5. Consider external variables (temperature, wind) for better predictions

## FILES GENERATED
1. 01_timeseries_raw_pollutants.html - Raw time series visualization
2. 02_stl_decomposition_pm25.html - STL decomposition plots
3. 03_adf_test_results.html - Stationarity test results
4. 04_acf_pacf_analysis.html - Autocorrelation analysis
5. 05_seasonality_patterns.html - Monthly & weekly patterns
6. 06_hourly_patterns.html - Diurnal variations
7. 07_heatmap_hour_vs_day.html - Hour vs day heatmap
8. 08_monthly_trends.html - Monthly trend analysis
9. 09_moving_averages.html - Moving average smoothing
10. adf_test_results.csv - ADF test statistics
11. hourly_patterns.csv - Hourly average data
12. monthly_patterns.csv - Monthly average data
"""

In [ ]:
# Save report
with open(f'{RESULTS_TIMESERIES}/TIMESERIES_ANALYSIS_REPORT.txt', 'w') as f:
    f.write(ts_report)

print("✅ Saved: TIMESERIES_ANALYSIS_REPORT.txt")
print(ts_report)

# FINAL SUMMARY

In [ ]:
print("✅ NOTEBOOK 03 (TIME SERIES ANALYSIS) COMPLETE!")

print(f"""
📊 TIME SERIES ANALYSIS COMPLETED:
   ✓ Raw time series visualization
   ✓ STL Decomposition (trend, seasonal, residual)
   ✓ Stationarity Testing (ADF test)
   ✓ ACF & PACF Analysis
   ✓ Seasonality Detection (monthly, weekly, hourly)
   ✓ Hourly Patterns Analysis
   ✓ Hour vs Day Heatmap
   ✓ Monthly Trends Analysis
   ✓ Moving Averages (7-day, 30-day)

💾 FILES SAVED ({RESULTS_TIMESERIES}/):
   ✓ 01_timeseries_raw_pollutants.html
   ✓ 02_stl_decomposition_pm25.html
   ✓ 03_adf_test_results.html
   ✓ 04_acf_pacf_analysis.html
   ✓ 05_seasonality_patterns.html
   ✓ 06_hourly_patterns.html
   ✓ 07_heatmap_hour_vs_day.html
   ✓ 08_monthly_trends.html
   ✓ 09_moving_averages.html
   ✓ adf_test_results.csv
   ✓ hourly_patterns.csv
   ✓ monthly_patterns.csv
   ✓ TIMESERIES_ANALYSIS_REPORT.txt

🎨 VISUALIZATIONS:
   ✓ Professional time series plots
   ✓ Beautiful decomposition visualizations
   ✓ Clear statistical test results
   ✓ Interactive seasonal patterns
   ✓ Heatmaps untuk temporal patterns

🚀 NEXT PHASE:
   → Notebook 04: Geospatial Analysis (Maps, Clustering, Kriging)

""")

print("✅ Ready for Geospatial Analysis!")